# 01 Exploratory data analysis

Role 2. Due Thursday 19 Nov.

What a pricing manager would want to know about the panel, and the graded point "Exploratory Data Analysis". Train rows only from step 3 on; test is never looked at.

Class notebooks followed: the data-loading and EDA sections of `CTrees_titanic` and `RF_risk`.

Every step is coded and runs top to bottom on the shared panel (about 1 minute). Your job: run it, read each output,
write the interpretation cells in your own words, and be able to explain every line (the professor can ask
anyone). Where a step says **Your reading**, replace the prompt with what the output shows.

Rules: validation only (`partition == "valid"`); test is opened once, in `99_final`. Shared code comes from `ml2`
and is never copied here. Figures are saved to `reports/figures/01_*.png`. Paste the lines printed at the
end into `RUN_LOG.md`. Clear the outputs before you commit (the data is licensed: no rows in Git).

## 1. Load and check

One row is one bond on one month-end snapshot date (a bond-month).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ml2.config import load_config, resolve
from ml2.data import check_panel, load_column_dictionary, load_panel
from ml2.features import categorical_columns, numeric_columns
from ml2.schema import feature_columns

cfg = load_config()
TARGET = cfg["problem"]["target"]
PREFIX = "01"
FIG = resolve("reports/figures")
FIG.mkdir(parents=True, exist_ok=True)
MATURITY_ORDER = ["1-3y", "3-5y", "5-7y", "7-10y", "10-20y", "20y+"]

def save(fig, name):
    """Save a figure to reports/figures/<prefix>_<name>.png."""
    fig.savefig(FIG / f"{PREFIX}_{name}.png", dpi=150, bbox_inches="tight")


check_panel()
df = load_panel()
print(df.shape, f"{len(feature_columns())} features")
pd.crosstab(df["partition"], df["grade_bucket"], margins=True)

In [ ]:
# From here on: train, gap and valid only. Test stays closed until the final notebook.
seen = df[df.partition != "test"]
train = df[df.partition == "train"]
per_snapshot = seen.groupby("snapshot_date").agg(
    partition=("partition", "first"), rows=("cusip_id", "size"),
    bonds=("cusip_id", "nunique"), issuers=("issuer_ticker", "nunique"))
per_snapshot

**Your reading.** How many bonds and companies per month-end, and is the panel stable over time?

## 2. Missing values

Share of missing values per feature. Nothing is filled here: the shared preprocessor imputes inside each model (median plus a missing flag).

In [ ]:
feats = feature_columns()
missing = pd.DataFrame({
    "train": train[feats].isna().mean(),
    "valid": seen.loc[seen.partition == "valid", feats].isna().mean(),
}).sort_values("train", ascending=False)
missing = missing[missing.max(axis=1) > 0]

fig, ax = plt.subplots(figsize=(8, 0.3 * len(missing) + 1))
missing[::-1].plot.barh(ax=ax)
ax.set_xlabel("Share of rows missing")
ax.set_title("Missing values by feature, train and valid")
save(fig, "missing")
missing.round(3)

**Your reading.** Which features have the most gaps, and why? (For example: no listed stock, so no volatility or market value; no published balance sheet, so no leverage. The 46 companies whose stock could not be confirmed as the bond issuer have their stock data left empty on purpose.)

## 3. The target

`spread_bps`: bond yield minus the Treasury yield of the same maturity, in basis points. Train rows only.

In [ ]:
print(train.groupby("grade_bucket")[TARGET].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(1))
print("skewness:", train.groupby("grade_bucket")[TARGET].skew().round(2).to_dict())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
cap = train[TARGET].quantile(0.995)
for ax, g in zip(axes[:2], ["IG", "HY"]):
    s = train.loc[train.grade_bucket == g, TARGET]
    ax.hist(s.clip(upper=cap), bins=80)
    ax.axvline(s.median(), color="black", ls="--", label=f"median {s.median():.0f} bp")
    ax.set_title(f"Spread, {g} (train)"); ax.set_xlabel("Spread (bp)"); ax.set_ylabel("Bond-months"); ax.legend()
axes[2].hist(np.log(train[TARGET].clip(lower=1)), bins=80)
axes[2].set_title("log(spread), all grades (train)"); axes[2].set_xlabel("log of spread in bp")
save(fig, "target")

**Your reading.** Is the spread skewed? Should the models predict log(spread) or the spread itself? The gate is measured in bp MAE, so the team must agree and log the decision in RUN_LOG.

## 4. What drives spreads

In [ ]:
heat = train.pivot_table(index="maturity_bucket", columns="grade_bucket", values=TARGET, aggfunc="median")
heat = heat.loc[MATURITY_ORDER, ["IG", "HY"]]

fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(heat.values, cmap="viridis")
ax.set_xticks(range(heat.shape[1]), heat.columns); ax.set_yticks(range(heat.shape[0]), heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.values[i, j]:.0f}", ha="center", va="center", color="white")
ax.set_xlabel("Grade"); ax.set_ylabel("Maturity bucket")
ax.set_title("Median spread (bp), train")
fig.colorbar(im, ax=ax, label="bp")
save(fig, "heatmap_maturity_grade")
heat.round(0)

In [ ]:
by_sector = train.groupby(["sector", "grade_bucket"])[TARGET].median().unstack()[["IG", "HY"]]
by_sector = by_sector.sort_values("IG")
fig, ax = plt.subplots(figsize=(8, 6))
by_sector.plot.barh(ax=ax)
ax.set_xlabel("Median spread (bp)"); ax.set_title("Median spread by sector and grade, train")
save(fig, "sector")
by_sector.round(0)

In [ ]:
pts = train.sample(5000, random_state=42)
fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
for ax, col in zip(axes, ["vol_252d", "leverage", "naive_dd", "log_par_volume"]):
    for g, colour in [("IG", "tab:blue"), ("HY", "tab:orange")]:
        p = pts[pts.grade_bucket == g]
        ax.scatter(p[col], p[TARGET].clip(lower=1), s=4, alpha=0.4, color=colour, label=g)
    ax.set_yscale("log"); ax.set_xlabel(col); ax.set_title(f"Spread against {col}")
axes[0].set_ylabel("Spread (bp, log scale)"); axes[0].legend()
save(fig, "scatter_drivers")

**Your reading.** One sentence of business interpretation per chart: longer maturity and lower grade pay more; which sectors are wide; does higher stock volatility or leverage go with wider spreads, and does distance to default go the other way?

## 5. The market over time

Median spread per snapshot next to the high-yield index spread (`hy_oas_bps`). Train, gap and valid only.

In [ ]:
over_time = seen.groupby(["snapshot_date", "grade_bucket"])[TARGET].median().unstack()[["IG", "HY"]]
oas = seen.groupby("snapshot_date")["hy_oas_bps"].median()

fig, ax = plt.subplots(figsize=(11, 4))
over_time.plot(ax=ax, marker="o")
ax2 = ax.twinx()
oas.plot(ax=ax2, color="grey", ls="--", label="HY index spread (right)")
for part, colour in [("gap", "lightgrey"), ("valid", "lightyellow")]:
    d = per_snapshot.index[per_snapshot.partition == part]
    if len(d):
        ax.axvspan(d.min(), d.max(), color=colour, alpha=0.6, label=part)
ax.set_ylabel("Median bond spread (bp)"); ax2.set_ylabel("HY index spread (bp)")
ax.set_xlabel("Snapshot date"); ax.set_title("The market over time: train, gap and valid")
ax.legend(loc="upper left"); ax2.legend(loc="upper right")
save(fig, "market_over_time")

**Your reading.** Did spreads move between the training months and validation? This is why the split is chronological and why a pricing rule frozen at the training months goes stale.

## 6. Correlations

In [ ]:
num = [c for c in numeric_columns() if train[c].nunique() > 2]
corr = train[num].corr()

fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(corr.values, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num)), num, rotation=90, fontsize=7); ax.set_yticks(range(len(num)), num, fontsize=7)
ax.set_title("Correlation of the numeric features, train")
fig.colorbar(im, ax=ax)
save(fig, "correlations")

upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack()
high = upper[upper.abs() > 0.8].sort_values(key=abs, ascending=False)
high.round(2).rename("correlation").to_frame()

**Your reading.** Which pairs are above 0.8? Tell the Naive Bayes owner (it assumes independent features) and keep them in mind when reading feature importances.

## 7. Data quality

Report problems; do not silently drop rows.

In [ ]:
# Spot-check 10 companies: does the issuer name match the ticker's company?
names = train.drop_duplicates("issuer_ticker")[["issuer_ticker", "issuer_name", "sector", "grade_bucket"]]
names.sample(10, random_state=1)

In [ ]:
# The 10 widest and 10 narrowest spreads per grade
cols = ["snapshot_date", "issuer_name", "years_to_maturity", "coupon_pct", "price", TARGET]
for g in ["IG", "HY"]:
    t = train[train.grade_bucket == g]
    print(f"--- {g}: widest"); print(t.nlargest(10, TARGET)[cols].to_string(index=False))
    print(f"--- {g}: narrowest"); print(t.nsmallest(10, TARGET)[cols].to_string(index=False))

In [ ]:
# Sector labels come from two classification schemes (for example "Health Care" and "Healthcare")
train["sector"].value_counts()

**Your reading.** Are the extreme spreads plausible (distressed companies, bonds near maturity, negative spreads on very short or very safe bonds)? Note the two sector naming schemes and the Unknown sector.

## Findings

Half a page for the technical annex and 3 to 4 bullets for the report, written from the outputs above. Figures for
slide 6 are in `reports/figures/01_*.png`.